# Preprocessing and Feature Engineering: Dwelling Permit Data

This notebook is for preprocessing and feature engineering our permit data.

## Preliminaries

In [1]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = 'notebook_connected' # For plotly graphs to render in this environment

import geopandas as gpd
from shapely.geometry import Point, Polygon
import json

In [2]:
# Set directory

PATH = "C:/Users/emshe/Desktop/BRAINSTATION/CAPSTONE/GIT_REPO/DATA/PERMITS"

In [3]:
# Define function to examine dataframes

def examine_df(name,df):
    """
    Check basic info about a dataframe df
    """
    
    print(f"\n\nNumber of records in the {name} is: {len(df)}\n")
    print(f"The columns in the {name} are: {df.columns}\n")
    print(f"\n Other info about {name}:")
    display(df.info())
    print(f"\n\nSample of records in the {name}:")
    display(df.head(5))

In [4]:
# Define function to map geographic boundaries for some geographic dataframe

def map_boundaries(gdf, name_col, geo_col, title):
    
    """
    Map geographic regions with boundaries
    """
    
    # Prepare geographic dataframe for conversion into json format
    new_gdf = gdf.copy()    
    new_gdf = new_gdf.set_geometry(geo_col)
    new_gdf['dummy'] = [i for i in range(len(new_gdf))]
    new_gdf = new_gdf.to_crs("EPSG:4326")

    # Use geopandas to generate a valid geojson
    geojson = new_gdf.set_index(name_col).__geo_interface__
    geojson = json.loads(new_gdf.to_json())
    
    # Plot
    fig = px.choropleth_map(
                data_frame = new_gdf,
                   geojson = geojson,
                 locations = name_col,
                     color = "dummy",
              featureidkey = f"properties.{name_col}",  # GeoJSON path
                 map_style = "carto-positron",
                    center = {"lat": 49.25, "lon": -123.1},
                      zoom = 8,
                   opacity = 0.6,
                     title = title
    )
    fig.update_traces(marker_line_width=0.5, marker_line_color='black')
    # fig.update_layout(
    #     geo=dict(fitbounds="locations", visible=False),
    #     margin={"r": 0, "t": 30, "l": 0, "b": 0}
    # )
    
    fig.update_layout(margin={"r":0,"t":30,"l":0,"b":0})
    
    fig.show()


In [5]:
# Load CPI csv file and set up inflation conversion dictionary

cpi_rows = pd.read_csv(
    "C:/Users/emshe/Desktop/BRAINSTATION/CAPSTONE/GIT_REPO/DATA/ECONOMIC/INFLATION/bc_cpi_excl_shelter.csv",
    header=None,
    skiprows=8,     # start at row 9 (0-based)
    nrows=5,        # read a few rows to grab 9 and 11
    usecols=range(1, 10)  # columns B to J (1–9)
)

# Extract years from row index 1
years = cpi_rows.iloc[1].astype(int).values

# Extract corresponding CPI values from row index 3
cpi_values = cpi_rows.iloc[3].astype(float).values

# Compute CPI_2024
cpi_2024 = cpi_values[-1]

# Build dictionary: {year: CPI_2024 / CPI_year}
inflation_dict = {
    year: cpi_2024 / cpi for year, cpi in zip(years, cpi_values)
}

print(inflation_dict)

{2016: 1.2172523961661341, 2017: 1.1952941176470588, 2018: 1.1660290742157615, 2019: 1.1390134529147982, 2020: 1.133085501858736, 2021: 1.1099781500364165, 2022: 1.041695146958305, 2023: 1.0106100795755968, 2024: 1.0}


In [6]:
# Load data

permits_df = pd.read_csv(f"{PATH}/issued_building_permits_filter_dwelling_purposes_cleaned.csv",
                        parse_dates=["IssueDate"])
permits_df_og = permits_df.copy()

# Convert 'Geom' column from JSON-like strings into Point geometries
permits_df["Geom"] = permits_df["Geom"].apply(parse_geom)

# Convert permits_df into a GeoDataFrame with 'Geom' as the geometry column
permits_gdf = gpd.GeoDataFrame(permits_df, geometry="Geom", crs="EPSG:4326")

examine_df("permits dataframe",permits_gdf)

NameError: name 'parse_geom' is not defined

In [147]:
# Load geographic GEOJSON file

nbhds_gdf = gpd.read_file("C:/Users/emshe/Desktop/BRAINSTATION/CAPSTONE/GIT_REPO/DATA/GEOGRAPHIC/PROCESSED/nbhds_with_zones.geojson")
if nbhds_gdf.crs != 'EPSG:3857':
    print("Setting CRS to EPSG:3857...")
    nbhds_gdf = nbhds_gdf.set_crs('EPSG:3857', allow_override=True)
print(nbhds_gdf.crs)  # Check the CRS after loading
examine_df('geographic dataframe',nbhds_gdf)

Setting CRS to EPSG:3857...
EPSG:3857


Number of records in the geographic dataframe is: 68

The columns in the geographic dataframe are: Index(['nbhd', 'zone', 'geometry'], dtype='object')


 Other info about geographic dataframe:
<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 68 entries, 0 to 67
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype   
---  ------    --------------  -----   
 0   nbhd      68 non-null     object  
 1   zone      68 non-null     object  
 2   geometry  68 non-null     geometry
dtypes: geometry(1), object(2)
memory usage: 1.7+ KB


None



Sample of records in the geographic dataframe:


,nbhd,zone,geometry
0,West End/Stanley Park North,West End/Stanley Park,"POLYGON ((-13707903.991 6324277.893, -13707728..."
1,West End/Stanley Park South,West End/Stanley Park,"POLYGON ((-13709954.808 6328768.906, -13708979..."
2,English Bay,English Bay,"POLYGON ((-13708084.955 6323746.439, -13707920..."
3,Downtown Central,Downtown,"POLYGON ((-13706769.27 6325491.913, -13706673...."
4,North False Creek,Downtown,"POLYGON ((-13705341.854 6321743.569, -13705338..."


## Apply inflation conversion

In [24]:
# Define function for inflation adjustment

def inflation_adjustment(df, inflation_dict):
    
    """
    Adjusts ProjectValue based on inflation rates.
    """
    
    # Ensure datetime
    if not pd.api.types.is_datetime64_any_dtype(df["IssueDate"]):
        raise ValueError("IssueDate must be a datetime column.")

    # Extract year
    df = df.copy()
    df["Year"] = df["IssueDate"].dt.year

    # Apply conversion
    df["ProjectValue_Adj"] = df.apply(
        lambda row: row["ProjectValue"] * inflation_dict.get(row["Year"], 1),
        axis=1
    )

    # Drop the original ProjectValue column
    df.drop(columns=["ProjectValue","Year"], inplace=True)
    
    return df

## Add geographic columns

In [91]:
# Function to convert Geom column from JSON to Point geometries

def parse_geom(geom_str):
    '''
    Convert 'Geom' column to correct format
    '''
    
    try:
        geom_json = json.loads(geom_str)
        return Point(geom_json["coordinates"])
    except Exception as e:
        print(f"Error parsing geometry: {e}")
        return None

In [122]:
# Define function to add zone and nbhd columns based on geography

def add_geo_cols(permits_gdf, nbhds_gdf):

    '''
    Add neighborhood and zone columns to permits df based on geographic coordinates
    '''

    # Remove invalid geometries (Surrey is invalid currently)

    # nbhds_gdf = nbhds_gdf[nbhds_gdf.geometry.type == 'Polygon']  # Filter out MultiPolygon geometries
    # nbhds_gdf = nbhds_gdf[nbhds_gdf.is_valid]  # Ensure all remaining geometries are valid
    
    permits_gdf = permits_gdf.copy()
    # Perform spatial join to determine which neighborhood and zone each permit belongs to
    new_df = gpd.sjoin(permits_gdf, nbhds_gdf, how="left", predicate="within")

    new_df.drop(columns = ['index_right'],inplace = True)
    
    return new_df

## Rename columns

In [184]:
# Create function to rename columns


def rename_columns(df):
    '''
    Rename columns of a dataframe manually to add underscores between words.
    '''
    # Manually rename columns with underscores and lowercase
    df.columns = [
        'issue_date', 
        'type_of_work', 
        'project_description', 
        'permit_category', 
        'specific_use_category', 
        'geom', 
        'project_value',  # Remove 'adj' from 'project_value_adj'
        'nbhd', 
        'zone'
    ]
    
    return df

## Expand specific_use_category column

In [195]:
# Define function to transform property_use column

def transform_specific_use_category(df):
    
    '''
    Transform the 'specific_use_category' column into binary columns for each unique use type
    '''
    
    # Create a list of unique use types
    unique_uses = set()
    for use in df['specific_use_category']:
        # Split by ', ' and add each use type (ignoring 'Dwelling Uses')
        use_types = [u.strip().lower().replace(" ", "_") for u in use.split(',')]
        unique_uses.update(use_types)
    
    # Create binary columns for each use type
    for use in unique_uses:
        df[use] = df['specific_use_category'].apply(lambda x: 1 if use in [u.strip().lower().replace(" ", "_") for u in x.split(',')] else 0)
    
    # Drop the original 'PropertyUse' column
    df.drop(columns=['specific_use_category'], inplace=True)

    binary_columns = df.columns.difference(['issue_date', 'type_of_work', 'project_description', 
                                            'permit_category', 'geom', 'project_value', 'nbhd', 'zone'])

    for col in binary_columns:
        non_zero_count = df[col].sum()
        total_count = len(df)
        
        # Calculate the percentage of non-zero entries
        non_zero_percentage = non_zero_count / total_count
        
        # Drop columns with less than 1% non-zero entries
        if non_zero_percentage < 0.01:
            df.drop(columns=[col], inplace=True)
    
    return df

## Create categorical dummy columns

In [213]:
# Define function for dummy columns

def create_dummy_columns(df):
    
    '''
    Creates dummy columns for 'type_of_work' and 'permit_category' columns.
    Drops the 'unknown' category for 'permit_category' and 'Addition / Alteration' for 'type_of_work'.
    '''
    
    # Create dummy columns for 'type_of_work' and 'permit_category'
    type_of_work_dummies = pd.get_dummies(df['type_of_work'], prefix='type_of_work', drop_first=False,dtype = int)
    permit_category_dummies = pd.get_dummies(df['permit_category'], prefix='permit_category', drop_first=False, dtype = int)

    # Drop 'unknown' category for 'permit_category' and 'Addition / Alteration' for 'type_of_work'
    permit_category_dummies = permit_category_dummies.drop(columns=['permit_category_unknown','permit_category_Renovation - Commercial/ Mixed Use - Lower Complexity'], errors='ignore')
    type_of_work_dummies = type_of_work_dummies.drop(columns=['type_of_work_Addition / Alteration'], errors='ignore')

    # Concatenate the dummy columns with the original DataFrame
    df = pd.concat([df, type_of_work_dummies, permit_category_dummies], axis=1)
    
    # Drop the original 'type_of_work' and 'permit_category' columns
    df.drop(columns=['type_of_work', 'permit_category'], inplace=True)

    # Clean column names
    df.columns = df.columns.str.replace(r'[\s\-\/]', '_', regex=True).str.lower()
    df.columns = df.columns.str.replace('___', '_', regex=True).str.lower()
    
    return df

## Apply all preprocessing to dataframe

In [201]:
# Define function to streamline all preprocessing

def preprocess():
    
    '''
    Load and apply all preprocessing functions to permits dataframe
    '''

    permits_df = pd.read_csv(f"{PATH}/issued_building_permits_filter_dwelling_purposes_cleaned.csv",
                        parse_dates=["IssueDate"])

    nbhds_gdf = gpd.read_file("C:/Users/emshe/Desktop/BRAINSTATION/CAPSTONE/GIT_REPO/DATA/GEOGRAPHIC/PROCESSED/nbhds_with_zones.geojson")
    nbhds_gdf = nbhds_gdf.set_crs('EPSG:3857', allow_override=True)
    nbhds_gdf = nbhds_gdf.to_crs(epsg=4326)
    
    
    # Convert 'Geom' column from JSON-like strings into Point geometries
    permits_df["Geom"] = permits_df["Geom"].apply(parse_geom)

    # Convert permits_df into a GeoDataFrame with 'Geom' as the geometry column
    permits_gdf = gpd.GeoDataFrame(permits_df, geometry="Geom", crs="EPSG:4326")
    
    # Adjust for inflation
    permits_gdf = inflation_adjustment(permits_gdf,inflation_dict)

    # Add geographic columns
    joined_gdf = add_geo_cols(permits_gdf,nbhds_gdf)
    permits_gdf = joined_gdf

    # Rename all columns
    permits_gdf = rename_columns(permits_gdf)

    # Expand specific_use_category column into binary columns
    permits_gdf = transform_specific_use_category(permits_gdf)

    # Replace other categorical columns with dummy columns
    permits_gdf = create_dummy_columns(permits_gdf)
    
    return joined_gdf, permits_gdf, nbhds_gdf

In [207]:
# Check the number of 1s in each binary specific use category column
binary_columns = permits_gdf.columns.difference(['issue_date', 'type_of_work', 'project_description', 'permit_category',
       'geom', 'project_value', 'nbhd', 'zone'])  # List binary columns


# Print the count of 1s in each binary column
for col in binary_columns:
    print(f"Column: {col}, Count of 1s: {permits_gdf[col].sum()}")

Column: duplex, Count of 1s: 1499
Column: duplex_w_secondary_suite, Count of 1s: 681
Column: dwelling_unit, Count of 1s: 598
Column: laneway_house, Count of 1s: 3780
Column: multiple_conversion_dwelling, Count of 1s: 342
Column: multiple_dwelling, Count of 1s: 5364
Column: permit_category_new_build___low_density_housing, Count of 1s: 6554
Column: permit_category_new_build___standalone_laneway, Count of 1s: 2015
Column: permit_category_renovation___commercial__mixed_use___lower_complexity, Count of 1s: 107
Column: permit_category_renovation___residential___lower_complexity, Count of 1s: 7896
Column: single_detached_house, Count of 1s: 8125
Column: single_detached_house_w_sec_suite, Count of 1s: 4626
Column: type_of_work_demolition___deconstruction, Count of 1s: 5714
Column: type_of_work_new_building, Count of 1s: 9465


In [219]:
# Sort dataframe

permits_gdf.sort_values(by = 'issue_date')

,issue_date,project_description,geom,project_value,nbhd,zone,duplex_w_secondary_suite,laneway_house,duplex,multiple_conversion_dwelling,dwelling_unit,multiple_dwelling,single_detached_house,single_detached_house_w_sec_suite,type_of_work_demolition_deconstruction,type_of_work_new_building,permit_category_new_build_low_density_housing,permit_category_new_build_standalone_laneway,permit_category_renovation_residential_lower_complexity
9420,2017-01-03,Low Density Housing - Demolition / Deconstruct...,POINT (-123.10509 49.25123),17929.411765,Riley Park,Mount Pleasant/Renfrew Heights,0,0,0,0,0,0,1,0,1,0,0,0,0
6533,2017-01-03,Field Review - Addition / Alteration - Interio...,POINT (-123.0941 49.2659),597.647059,Mount Pleasant,Mount Pleasant/Renfrew Heights,0,0,0,0,0,1,0,0,0,0,0,0,1
12303,2017-01-04,Low Density Housing - Demolition / Deconstruct...,POINT (-123.09243 49.24762),17929.411765,Riley Park,Mount Pleasant/Renfrew Heights,0,0,0,0,0,0,1,0,1,0,0,0,0
2206,2017-01-04,Field Review - Addition / Alteration - (#1203)...,POINT (-123.14077 49.28898),41835.294118,West End/Stanley Park North,West End/Stanley Park,0,0,0,0,0,1,0,0,0,0,0,0,1
6515,2017-01-04,Field Review - Addition / Alteration - (#702) ...,POINT (-123.13381 49.29217),47811.764706,Downtown Central,Downtown,0,0,0,0,0,1,0,0,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23257,2025-05-09,Low Density Housing - Addition / Alteration - ...,POINT (-123.20704 49.2653),190000.000000,Point Grey,Kitsilano/Point Grey,0,0,0,0,0,0,1,0,0,0,0,0,0
17598,2025-05-09,Low Density Housing - New Building - To constr...,POINT (-123.19114 49.25586),605700.000000,Westside/Kerrisdale Remainder,Westside/Kerrisdale,0,0,0,0,0,0,1,0,0,1,1,0,0
10750,2025-05-09,Low Density Housing - Demolition / Deconstruct...,POINT (-123.20485 49.2649),40000.000000,Point Grey,Kitsilano/Point Grey,0,0,0,0,0,0,1,0,1,0,0,0,0
14170,2025-05-09,Low Density Housing - New Building - To constr...,POINT (-123.20487 49.26488),350000.000000,Point Grey,Kitsilano/Point Grey,0,1,0,0,0,0,0,0,0,1,1,0,0


In [214]:
# Preprocess permits dataframe

joined_gdf, permits_gdf, nbhds_gdf = preprocess()

examine_df('permits dataframe',permits_gdf)



Number of records in the permits dataframe is: 25445

The columns in the permits dataframe are: Index(['issue_date', 'project_description', 'geom', 'project_value', 'nbhd',
       'zone', 'duplex_w_secondary_suite', 'laneway_house', 'duplex',
       'multiple_conversion_dwelling', 'dwelling_unit', 'multiple_dwelling',
       'single_detached_house', 'single_detached_house_w_sec_suite',
       'type_of_work_demolition_deconstruction', 'type_of_work_new_building',
       'permit_category_new_build_low_density_housing',
       'permit_category_new_build_standalone_laneway',
       'permit_category_renovation_residential_lower_complexity'],
      dtype='object')


 Other info about permits dataframe:
<class 'geopandas.geodataframe.GeoDataFrame'>
Index: 25445 entries, 0 to 25444
Data columns (total 19 columns):
 #   Column                                                   Non-Null Count  Dtype         
---  ------                                                   --------------  -----    

None



Sample of records in the permits dataframe:


,issue_date,project_description,geom,project_value,nbhd,zone,duplex_w_secondary_suite,laneway_house,duplex,multiple_conversion_dwelling,dwelling_unit,multiple_dwelling,single_detached_house,single_detached_house_w_sec_suite,type_of_work_demolition_deconstruction,type_of_work_new_building,permit_category_new_build_low_density_housing,permit_category_new_build_standalone_laneway,permit_category_renovation_residential_lower_complexity
0,2017-04-12,Low Density Housing - New Building - To constr...,POINT (-123.04389 49.25452),250115.294118,Renfrew,Mount Pleasant/Renfrew Heights,0,0,0,0,0,0,0,1,0,1,1,0,0
1,2022-11-14,Low Density Housing - Demolition / Deconstruct...,POINT (-123.06777 49.22493),15625.427204,Fraser View/Killarny,Southeast Vancouver,0,0,0,0,0,0,1,0,1,0,0,0,0
2,2017-08-23,Field Review - Demolition / Deconstruction - T...,POINT (-123.12117 49.25887),75669.289412,South Granville,South Granville/Oak,0,0,0,0,0,1,0,0,1,0,0,0,0
3,2017-09-28,Low Density Housing - New Building - To constr...,POINT (-123.08467 49.23656),183866.117647,Sunset,Southeast Vancouver,0,1,0,0,0,0,0,0,0,1,0,1,0
4,2018-08-03,Low Density Housing - Demolition / Deconstruct...,POINT (-123.08949 49.23941),17490.436113,Sunset,Southeast Vancouver,0,0,0,0,0,0,0,1,1,0,0,0,0


In [215]:
# Save preprocessed dataframe
permits_gdf.to_csv("C:/Users/emshe/Desktop/BRAINSTATION/CAPSTONE/GIT_REPO/DATA/PERMITS/issued_building_permits_filter_dwelling_purposes_preprocessed.csv", index=False)

# Increase project value minimum

In [8]:
# Load permits

preproc_permits = pd.read_csv(f'{PATH}/issued_building_permits_filter_dwelling_purposes_preprocessed.csv',
                parse_dates=['issue_date']) # Force datetime format for date columns

In [9]:
# Set new project value minimum

new_permits = preproc_permits[preproc_permits['project_value']>500]

In [12]:
# Save preprocessed dataframe
new_permits.to_csv("C:/Users/emshe/Desktop/BRAINSTATION/CAPSTONE/GIT_REPO/DATA/PERMITS/issued_building_permits_filter_dwelling_purposes_preprocessed.csv", index=False)